# Versao 11 - Pre-Processamento

Este notebook explica detalhadamente o que muda entre `versao10` e `versao11`. A rede permanece a mesma; o pipeline de entrada e que fica mais seletivo.

## Artefatos Da Ultima Execucao

A configuracao corrigida da `versao11` passa a registrar:

- features removidas por `null_pct = 100%`: `9`
- series analisadas no controle de qualidade: `2228`
- series mantidas para o `split`: `2228`
- series descartadas por filtro de `state`: `0`
- tamanhos dos splits: `train = 1559`, `validation = 334`, `test = 335`
- series de treino com foco observacional por `class`: `1143`
- series em que o foco por `class` foi aplicado de fato: `1112`
- series de treino que exigem `fallback` para a serie completa: `31`

Leitura metodologica: a `versao11` nao altera a arquitetura da `versao10`. O experimento muda o que entra no treinamento. Primeiro, remove do conjunto de entrada as features completamente vazias. Depois, no treino das classes `1..9`, tenta remover observacoes normais usando o `class` observacional, mantendo observacoes de erro (`1..9`) ou transientes (`101..109`).


## Regras novas da versao 11

A preparacao passa a impor tres regras adicionais:

1. **limpeza estrutural de features**: variaveis com `null_pct = 100%` sao removidas do modelo por nao carregarem informacao;
2. **foco por `class` observacional no treino**: para series de classe global `1..9`, o treino tenta manter apenas observacoes cujo `class` indique erro ou transiente;
3. **viabilidade do split estratificado**: se uma classe ficar com poucas series depois das etapas anteriores, ela pode ser descartada automaticamente para manter consistencia entre treino, validacao e teste.

Aqui, o filtro de treino usa o proprio `class` observacional:

- `NaN` -> nao entra no foco observacional
- `0` -> normal, removido do foco das falhas
- `1..9` -> erro, mantido
- `101..109` -> transiente, mantido

Ja o `state` continua existindo como rotulo auxiliar por observacao, mas nao e mais usado como criterio de recorte nesta versao.


In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao11" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import versao11.pipeline_v11 as pipeline_v11

pipeline_v11 = importlib.reload(pipeline_v11)
IGNORE_INDEX = pipeline_v11.IGNORE_INDEX
FULL_FEATURE_COLUMNS = pipeline_v11.FULL_FEATURE_COLUMNS
STATE_SENSOR_COLUMNS = pipeline_v11.STATE_SENSOR_COLUMNS
OBSERVATION_CLASS_CODES = pipeline_v11.OBSERVATION_CLASS_CODES
ALL_NULL_FEATURE_COLUMNS = pipeline_v11.ALL_NULL_FEATURE_COLUMNS
SELECTED_FEATURE_COLUMNS = pipeline_v11.SELECTED_FEATURE_COLUMNS
require_classification_stack = pipeline_v11.require_classification_stack
_require_dataframe_stack = pipeline_v11._require_dataframe_stack
_map_training_state_phases = pipeline_v11._map_training_state_phases
v10 = pipeline_v11.v10


## Funcoes-chave Tornadas Explicitas
### Percentual de `NaN` nas features mantidas

In [2]:
def _compute_feature_nan_ratio(
    frame: pd.DataFrame,
    *,
    feature_columns: list[str] | None = None,
) -> float:
    feature_columns = feature_columns or SELECTED_FEATURE_COLUMNS
    matrix = (
        frame[feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=np.float64, copy=False)
    )
    if matrix.size == 0:
        return 1.0
    return float((~np.isfinite(matrix)).mean())

### Mapeamento do `class` observacional para o foco de treino


In [3]:
BASE_OBSERVATION_FAULT_CLASS_CODES = [code for code in OBSERVATION_CLASS_CODES if 1 <= int(code) <= 9]
TRANSIENT_OBSERVATION_CLASS_CODES = [code for code in OBSERVATION_CLASS_CODES if int(code) >= 100]
TRAINING_FOCUS_OBSERVATION_CLASS_CODES = [code for code in OBSERVATION_CLASS_CODES if int(code) != 0]

def _observation_class_code_from_value(value: object) -> int:
    if pd is not None and pd.isna(value):
        return IGNORE_INDEX
    try:
        return int(value)
    except (TypeError, ValueError):
        return IGNORE_INDEX


def _build_training_observation_class_mask(values: np.ndarray) -> np.ndarray:
    focus_codes = set(TRAINING_FOCUS_OBSERVATION_CLASS_CODES)
    mapped = [
        _observation_class_code_from_value(value) in focus_codes
        for value in np.asarray(values, dtype=object)
    ]
    return np.asarray(mapped, dtype=bool)


### Relatorio de features removidas

In [4]:
def build_feature_selection_report() -> pd.DataFrame:
    _require_dataframe_stack()
    rows = []
    for column_name in FULL_FEATURE_COLUMNS:
        dropped = column_name in ALL_NULL_FEATURE_COLUMNS
        rows.append(
            {
                "column": column_name,
                "null_pct": 100.0 if dropped else np.nan,
                "selected_for_modeling": not dropped,
                "selection_reason": "all_null_feature_removed" if dropped else "kept_for_modeling",
                "column_type": "state" if column_name in STATE_SENSOR_COLUMNS else "continuous",
            }
        )
    return pd.DataFrame(rows)

### Relatorio de qualidade das series

In [5]:
def build_series_quality_report(
    manifest: pd.DataFrame,
    *,
    max_nan_ratio: float = 0.70,
) -> pd.DataFrame:
    require_classification_stack()
    _require_dataframe_stack()

    rows = []
    for _, row in manifest.iterrows():
        frame = v10._prepare_raw_frame(row["file_path"])
        state_phases = _map_training_state_phases(frame["state"].to_numpy())
        observation_class_codes = np.asarray(
            [
                _observation_class_code_from_value(value)
                for value in frame["class"].to_numpy(dtype=object)
            ],
            dtype=np.int64,
        )

        nan_ratio = _compute_feature_nan_ratio(frame)
        n_rows_original = int(len(frame))
        n_normal_rows = int((state_phases == 0).sum())
        n_transient_rows = int((state_phases == 1).sum())
        n_failure_rows = int((state_phases == 2).sum())
        n_negative_rows = int(np.isin(state_phases, [1, 2]).sum())
        n_observation_class_zero_rows = int((observation_class_codes == 0).sum())
        n_observation_class_fault_rows = int(
            np.isin(observation_class_codes, BASE_OBSERVATION_FAULT_CLASS_CODES).sum()
        )
        n_observation_class_transient_rows = int(
            np.isin(observation_class_codes, TRANSIENT_OBSERVATION_CLASS_CODES).sum()
        )
        n_observation_class_focus_rows = int(
            np.isin(observation_class_codes, TRAINING_FOCUS_OBSERVATION_CLASS_CODES).sum()
        )

        keep_for_v11 = True
        drop_reason = ""

        rows.append(
            {
                "file_path": row["file_path"],
                "series_id": row["series_id"],
                "class_label_int": int(row["class_label_int"]),
                "source_type": row["source_type"],
                "n_rows_original": n_rows_original,
                "nan_ratio_selected_features": nan_ratio,
                "n_state_normal_rows": n_normal_rows,
                "n_state_transient_rows": n_transient_rows,
                "n_state_failure_rows": n_failure_rows,
                "n_negative_state_rows": n_negative_rows,
                "n_observation_class_zero_rows": n_observation_class_zero_rows,
                "n_observation_class_fault_rows": n_observation_class_fault_rows,
                "n_observation_class_transient_rows": n_observation_class_transient_rows,
                "n_observation_class_focus_rows": n_observation_class_focus_rows,
                "keep_for_v11": bool(keep_for_v11),
                "drop_reason": drop_reason,
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["keep_for_v11", "class_label_int", "series_id"],
        ascending=[False, True, True],
    ).reset_index(drop=True)


### Corte do treino por `class` observacional


In [6]:
def _prepare_frame_for_split(
    *,
    file_path: str | Path,
    global_class_label: int,
    split_name: str,
) -> tuple[pd.DataFrame, dict[str, object]]:
    frame = v10._prepare_raw_frame(file_path)
    nan_ratio = _compute_feature_nan_ratio(frame)
    n_rows_original = int(len(frame))

    training_focus_requested = str(split_name) == "train" and int(global_class_label) != 0
    training_focus_applied = False
    training_focus_fallback_reason = ""
    n_rows_removed_for_training_focus = 0

    if training_focus_requested:
        keep_mask = _build_training_observation_class_mask(frame["class"].to_numpy())
        filtered_frame = frame.loc[keep_mask].reset_index(drop=True)
        if filtered_frame.empty:
            training_focus_fallback_reason = "empty_after_observation_class_filter"
        else:
            n_rows_removed_for_training_focus = int(n_rows_original - len(filtered_frame))
            frame = filtered_frame
            training_focus_applied = True

    if training_focus_applied:
        preprocessing_mode = "observation_class_nonzero_only_for_fault_train"
    elif training_focus_fallback_reason:
        preprocessing_mode = "full_series_fallback_empty_after_class_filter"
    else:
        preprocessing_mode = "full_series"

    metadata = {
        "nan_ratio_selected_features": float(nan_ratio),
        "n_rows_original": n_rows_original,
        "n_rows_used": int(len(frame)),
        "n_rows_removed_for_training_focus": int(n_rows_removed_for_training_focus),
        "training_focus_requested": bool(training_focus_requested),
        "training_focus_applied": bool(training_focus_applied),
        "training_focus_fallback_reason": str(training_focus_fallback_reason),
        "negative_only_applied": bool(training_focus_applied),
        "preprocessing_mode": preprocessing_mode,
    }
    return frame, metadata


In [7]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao11" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import versao11.pipeline_v11 as pipeline_v11

pipeline_v11 = importlib.reload(pipeline_v11)
ALL_NULL_FEATURE_COLUMNS = pipeline_v11.ALL_NULL_FEATURE_COLUMNS
SELECTED_FEATURE_COLUMNS = pipeline_v11.SELECTED_FEATURE_COLUMNS
build_feature_selection_report = pipeline_v11.build_feature_selection_report
build_series_quality_report = pipeline_v11.build_series_quality_report
load_bundle = pipeline_v11.load_bundle
load_split_arrays = pipeline_v11.load_split_arrays
prepare_classification_artifacts = pipeline_v11.prepare_classification_artifacts

DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
RUN_NAME = "classificacao_v11_foco_por_class_observacional"

artifacts = prepare_classification_artifacts(
    dataset_root=DATASET_ROOT,
    run_name=RUN_NAME,
    random_state=42,
    sequence_length=180,
)
bundle = load_bundle(artifacts.bundle_path)

train_arrays = load_split_arrays(artifacts.split_npz_paths["train"])
validation_arrays = load_split_arrays(artifacts.split_npz_paths["validation"])
test_arrays = load_split_arrays(artifacts.split_npz_paths["test"])

print("Run dir:", artifacts.run_dir)
print("Bundle path:", artifacts.bundle_path)
print("Numero de colunas selecionadas:", len(bundle.selected_columns))
print("Numero de atributos em X_tab:", len(bundle.statistical_feature_names))
print("Features removidas por null_pct = 100%:", ALL_NULL_FEATURE_COLUMNS)
print("Mapeamento de fonte:", bundle.source_mapping)


Run dir: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v11/classificacao_v11_foco_por_class_observacional
Bundle path: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v11/classificacao_v11_foco_por_class_observacional/bundle_v11.json
Numero de colunas selecionadas: 18
Numero de atributos em X_tab: 162
Features removidas por null_pct = 100%: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']
Mapeamento de fonte: {'well': 0, 'simulated': 1, 'drawn': 2}


In [8]:
feature_report = build_feature_selection_report()
quality_report = pd.read_csv(Path(artifacts.run_dir) / "series_quality_report.csv")
train_metadata = pd.read_csv(Path(artifacts.run_dir) / "train_metadata.csv")
validation_metadata = pd.read_csv(Path(artifacts.run_dir) / "validation_metadata.csv")
test_metadata = pd.read_csv(Path(artifacts.run_dir) / "test_metadata.csv")

display(feature_report)

drop_reason_series = quality_report["drop_reason"] if "drop_reason" in quality_report.columns else pd.Series([""] * len(quality_report))
focus_rows_series = (
    quality_report["n_observation_class_focus_rows"]
    if "n_observation_class_focus_rows" in quality_report.columns
    else pd.Series([0] * len(quality_report))
)

resumo_qualidade = pd.DataFrame(
    [
        {
            "series_totais": len(quality_report),
            "series_mantidas_para_split": int(quality_report["keep_for_v11"].sum()),
            "series_descartadas_split_inviavel": int((drop_reason_series == "too_few_series_for_stratified_split").sum()),
            "series_sem_foco_observacional": int((focus_rows_series == 0).sum()),
            "features_removidas": len(ALL_NULL_FEATURE_COLUMNS),
            "features_mantidas": len(SELECTED_FEATURE_COLUMNS),
        }
    ]
)
display(resumo_qualidade)
display(quality_report.head())

focus_requested_series = (
    train_metadata["training_focus_requested"]
    if "training_focus_requested" in train_metadata.columns
    else train_metadata["negative_only_applied"]
)
focus_applied_series = (
    train_metadata["training_focus_applied"]
    if "training_focus_applied" in train_metadata.columns
    else train_metadata["negative_only_applied"]
)
fallback_series = (
    train_metadata["training_focus_fallback_reason"]
    if "training_focus_fallback_reason" in train_metadata.columns
    else pd.Series([""] * len(train_metadata))
)
removed_when_applied = train_metadata.loc[focus_applied_series.astype(bool), "n_rows_removed_for_training_focus"]

resumo_treino = pd.DataFrame(
    [
        {
            "train_series": len(train_metadata),
            "train_foco_solicitado": int(focus_requested_series.sum()),
            "train_foco_aplicado": int(focus_applied_series.sum()),
            "train_fallback_serie_completa": int((fallback_series != "").sum()),
            "media_linhas_originais": float(train_metadata["n_rows_original"].mean()),
            "media_linhas_usadas": float(train_metadata["n_rows_used"].mean()),
            "media_linhas_removidas_quando_foco_aplicado": float(removed_when_applied.mean()) if not removed_when_applied.empty else 0.0,
        }
    ]
)
display(resumo_treino)


,column,null_pct,selected_for_modeling,selection_reason,column_type
0,ABER-CKGL,100.0,False,all_null_feature_removed,continuous
1,ABER-CKP,100.0,False,all_null_feature_removed,continuous
2,ESTADO-DHSV,NaN,True,kept_for_modeling,state
3,ESTADO-M1,NaN,True,kept_for_modeling,state
4,ESTADO-M2,NaN,True,kept_for_modeling,state
5,ESTADO-PXO,NaN,True,kept_for_modeling,state
6,ESTADO-SDV-GL,NaN,True,kept_for_modeling,state
7,ESTADO-SDV-P,NaN,True,kept_for_modeling,state
8,ESTADO-W1,NaN,True,kept_for_modeling,state
9,ESTADO-W2,NaN,True,kept_for_modeling,state


,series_totais,series_mantidas_para_split,series_descartadas_split_inviavel,series_sem_foco_observacional,features_removidas,features_mantidas
0,2228,2228,0,637,9,18


,file_path,series_id,class_label_int,source_type,n_rows_original,nan_ratio_selected_features,n_state_normal_rows,n_state_transient_rows,n_state_failure_rows,n_negative_state_rows,n_observation_class_zero_rows,n_observation_class_fault_rows,n_observation_class_transient_rows,n_observation_class_focus_rows,keep_for_v11,drop_reason,class_count_after_primary_filters,min_series_per_class_for_split
0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201010207,0,well,21474,0.0,17874,0,0,0,17874,0,0,0,True,NaN,594,7
1,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201060114,0,well,21527,0.0,17927,0,0,0,17927,0,0,0,True,NaN,594,7
2,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201110124,0,well,21517,0.0,17917,0,0,0,17917,0,0,0,True,NaN,594,7
3,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201160311,0,well,21410,0.0,17810,0,0,0,17810,0,0,0,True,NaN,594,7
4,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201210228,0,well,21453,0.0,17853,0,0,0,17853,0,0,0,True,NaN,594,7


,train_series,train_foco_solicitado,train_foco_aplicado,train_fallback_serie_completa,media_linhas_originais,media_linhas_usadas,media_linhas_removidas_quando_foco_aplicado
0,1559,1143,1112,1559,35277.453496,32015.085953,4573.768885


In [9]:
resumo_arrays = pd.DataFrame(
    [
        {
            "split": "train",
            "X_seq": train_arrays["X_seq"].shape,
            "X_tab": train_arrays["X_tab"].shape,
            "X_missing": train_arrays["X_missing"].shape,
            "X_frozen": train_arrays["X_frozen"].shape,
            "y": train_arrays["y"].shape,
            "y_step_class": train_arrays["y_step_class"].shape,
            "y_step_state": train_arrays["y_step_state"].shape,
        },
        {
            "split": "validation",
            "X_seq": validation_arrays["X_seq"].shape,
            "X_tab": validation_arrays["X_tab"].shape,
            "X_missing": validation_arrays["X_missing"].shape,
            "X_frozen": validation_arrays["X_frozen"].shape,
            "y": validation_arrays["y"].shape,
            "y_step_class": validation_arrays["y_step_class"].shape,
            "y_step_state": validation_arrays["y_step_state"].shape,
        },
        {
            "split": "test",
            "X_seq": test_arrays["X_seq"].shape,
            "X_tab": test_arrays["X_tab"].shape,
            "X_missing": test_arrays["X_missing"].shape,
            "X_frozen": test_arrays["X_frozen"].shape,
            "y": test_arrays["y"].shape,
            "y_step_class": test_arrays["y_step_class"].shape,
            "y_step_state": test_arrays["y_step_state"].shape,
        },
    ]
)
display(resumo_arrays)

,split,X_seq,X_tab,X_missing,X_frozen,y,y_step_class,y_step_state
0,train,"(1559, 180, 18)","(1559, 162)","(1559, 180, 18)","(1559, 180, 18)","(1559,)","(1559, 180)","(1559, 180)"
1,validation,"(334, 180, 18)","(334, 162)","(334, 180, 18)","(334, 180, 18)","(334,)","(334, 180)","(334, 180)"
2,test,"(335, 180, 18)","(335, 162)","(335, 180, 18)","(335, 180, 18)","(335,)","(335, 180)","(335, 180)"


## Leitura final

A `versao11` passa a separar dois problemas de forma mais coerente com o `3W`. Primeiro, remove do modelo variaveis que nao carregam nenhum sinal. Depois, no treino das falhas, usa o `class` observacional como criterio de foco, de modo que o modelo aprenda as classes `1..9` a partir dos trechos em que o proprio rotulo indica erro ou transiente.

Ja o `state` continua relevante como alvo auxiliar por observacao, mas nao e mais tratado como se fosse o rotulo do evento. Essa correcao deixa a hipotese da `versao11` alinhada com o significado real dos dados.
